In [ ]:
import datetime
import pandas as pd
import subprocess
import matplotlib.path as mplPath
import sys
import calendar
from calendar import monthrange

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import re


import ncutils as nc
import importlib

importlib.reload(nc)

In [ ]:


year=2004
#objn='MRG_Objects'

f0=f'/glade/derecho/scratch/hannay/MOAPP/moaap_output/monthly/{year:04}*_CAM7_L58_HR_ObjectMasks__dt-1h_MOAAP-masks.nc'
fobs=f'/glade/derecho/scratch/fudan/pub/MOAAP_ERA5/{year:04}*_ERA5_ObjectMasks__dt-1h_MOAAP-masks.nc'




Xc=xr.open_mfdataset( f0 ,data_vars='different', coords='different')
Xo=xr.open_mfdataset( fobs ,data_vars='different', coords='different')

In [ ]:
"""
allvars=['PR', 'MCS_Tb_Objects', 'BT', 'BT_Objects', 'FR', 'FR_Objects', 'T850', 'CY_Objects', 'ACY_Objects', 'SLP', 'TC_Objects', 'MS', 'MS_Objects', 
 'IVT', 'IVT_Objects', 'AR_Objects', 'CY_z500_Objects', 'ACY_z500_Objects', 'Z500', 'COL_Objects', 'JET_Objects', 'UV200', 'MRG_Objects', 'IGW_Objects', 
 'Kelvin_Objects', 'EIG0_Objects', 'ER_Objects', 'time', 'lat', 'lon']
"""
do_vars=[
        'MCS_Tb_Objects', 
        'IGW_Objects', 
        'Kelvin_Objects', 
        'EIG0_Objects', 
        'ER_Objects'
        'FR_Objects', 
        'CY_Objects', 
        'ACY_Objects', 
        'MS_Objects', 
        'IVT_Objects', 
        'AR_Objects', 
        'CY_z500_Objects', 
        'ACY_z500_Objects', 
        'COL_Objects', 
        'JET_Objects', 
        'BT_Objects', 
        'TC_Objects', 
        #'MRG_Objects', 
        ]

lato=Xo.lat.values
lono=Xo.lon.values
latc=Xc.lat.values
lonc=Xc.lon.values


nto,nyo,nxo=Xo.sizes['time'],Xo.sizes['yc'],Xo.sizes['xc']
ndo=nto//24

ntc,nyc,nxc=Xc.sizes['time'],Xc.sizes['yc'],Xc.sizes['xc']
ndc=ntc//24

for objn in do_vars:

    print( f" Doing {objn}" )
    objs_d_o= np.zeros( ( ndo, nyo, nxo ) )
    objprec_d_o= np.zeros( ( ndo, nyo, nxo ) )
    prec_d_o= np.zeros( ( ndo, nyo, nxo ) )
    
    for d in np.arange(ndo):
        # read in object masks and precip in one day chunks
        objs_o  = Xo[objn][d*24:d*24+24,:,:].values
        prec    = Xo['PR'][d*24:d*24+24,:,:].values
        # flatten obj mask to indicate only 1 or 0 (present / not present)
        boole   = (objs_o > 0)
        obj01   = boole.astype(int)
        # perform time averages to obtain daily means
        objs_d_o[d,:,:] = np.average( obj01 , axis=0)   # object 'presence' 
        objprec_d_o[d,:,:] = np.average( obj01 *prec, axis=0)  #precip from these objs 
        prec_d_o[d,:,:] = np.average( prec, axis=0)  # total precip
    
        print(f'{d}',end=',', flush=True)
    
    Xout1 = nc.wrt_objprec(year=year, exp='ERA5_Obs', lat=lato, lon=lono, objName=objn, objmsk=objs_d_o, objprec=objprec_d_o, prec=prec_d_o)
    
    objs_d_c= np.zeros( ( ndc, nyc, nxc ) )
    objprec_d_c= np.zeros( ( ndc, nyc, nxc ) )
    prec_d_c= np.zeros( ( ndc, nyc, nxc ) )
    
    for d in np.arange(ndc):
        # read in object masks and precip in one day chunks
        objs_c  = Xc[objn][d*24:d*24+24,:,:].values
        prec    = Xc['PR'][d*24:d*24+24,:,:].values
        # flatten obj mask to indicate only 1 or 0 (present / not present)
        boole   = (objs_c > 0)
        obj01 = boole.astype(int)
        # perform time averages to obtain daily means
        objs_d_c[d,:,:] = np.average( obj01, axis=0)    
        objprec_d_c[d,:,:] = np.average( obj01*prec, axis=0)    
        prec_d_c[d,:,:] = np.average( prec, axis=0)   
    
        print(f'{d}',end=',', flush=True)

    Xout2 = nc.wrt_objprec(year=year, exp='CAM7_HR', lat=latc, lon=lonc, objName=objn, objmsk=objs_d_c, objprec=objprec_d_c, prec=prec_d_c)
    print(f"next ...", flush=True)

In [ ]:
if (stupid == 'hegseth'):
    print('croak here')

In [ ]:
### Oh yeah, remember to define
stupid='hegseth'

In [ ]:
objs_d_o_tv=np.average( objs_d_o, axis=0 )
objs_d_c_tv=np.average( objs_d_c, axis=0 )
objprec_d_o_tv=np.average( objprec_d_o, axis=0 )
objprec_d_c_tv=np.average( objprec_d_c, axis=0 )

prec_d_o_tv=np.average( prec_d_o, axis=0 )
prec_d_c_tv=np.average( prec_d_c, axis=0 )




In [ ]:
plt.plot(prec_d_c_tv.flatten())

In [ ]:
workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.Plotting import LatLonMaps as LL
from myPythonTools.Utils import utils as uti



In [ ]:
flev=np.linspace( 0.,.2, num=51 )

flev=0.2*np.linspace( 0.,2.,num=51 )

fields=[objs_d_c_tv , objs_d_o_tv]
fields=[objprec_d_c_tv , objprec_d_o_tv]
lons=[lonc,lono]
lats=[latc,lato]
clevs=[flev,flev]
LL.Maps( fields=fields ,lons=lons ,lats=lats ,clevs=clevs, cmaps=['gist_ncar','gist_ncar'])

In [ ]:
objprec_d_o.shape

In [ ]:
Xout

In [ ]:
print( Xout.year.values)